In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from google.colab import drive
from sklearn.preprocessing import MinMaxScaler
import plotly.graph_objects as go

# =====================================================================
# 0. CONFIGURACIÓN Y MONTAJE DE GOOGLE DRIVE
# =====================================================================
print("🔌 Conectando con Google Drive...")
drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
os.makedirs(ruta_drive, exist_ok=True)
ruta_csv = os.path.join(ruta_drive, 'datos_series_temporales_deep_learning.csv')

# =====================================================================
# 1. GENERACIÓN DE DATASET TEMPORAL Y GUARDADO EN CSV
# =====================================================================
print("\n⏳ Creando dataset anual de generación solar horaria...")
np.random.seed(42)
fechas = pd.date_range(start="2025-01-01", end="2025-12-31 23:00:00", freq="h")
horas = fechas.hour

# Simulación física de curva solar con variación estacional y ruido meteorológico
radiacion_base = np.where((horas > 5) & (horas < 20), np.sin((horas - 5) / 14 * np.pi) * 500, 0)
variacion_estacional = 1 + np.sin(2 * np.pi * fechas.dayofyear / 365.0) * 0.4
ruido_nubes = np.random.normal(1, 0.2, len(fechas))
generacion_mw = np.clip(radiacion_base * variacion_estacional * ruido_nubes * 0.1, 0, None)

df_solar = pd.DataFrame({'Generacion_MW': generacion_mw}, index=fechas)
df_solar.to_csv(ruta_csv)
print(f"✅ Dataset original guardado exitosamente en: {ruta_csv}")

# =====================================================================
# 2. INGENIERÍA DE TENSORES (VENTANAS DESLIZANTES)
# =====================================================================
# El Deep Learning requiere normalización estricta para evitar la saturación de gradientes
escalador = MinMaxScaler(feature_range=(0, 1))
datos_normalizados = escalador.fit_transform(df_solar[['Generacion_MW']].values)

# Construcción de tensores: miramos 24 horas hacia el pasado para predecir la hora siguiente
def construir_ventanas_temporales(datos, ventana_pasada=24):
    X, y = [], []
    for i in range(len(datos) - ventana_pasada):
        X.append(datos[i:(i + ventana_pasada)])
        y.append(datos[i + ventana_pasada])
    return np.array(X), np.array(y)

X_numpy, y_numpy = construir_ventanas_temporales(datos_normalizados, ventana_pasada=24)

X_tensor = torch.tensor(X_numpy, dtype=torch.float32)
y_tensor = torch.tensor(y_numpy, dtype=torch.float32)

# División cronológica del set de datos (80% entrenamiento, 20% validación de producción)
limite = int(len(X_tensor) * 0.8)
X_train, X_test = X_tensor[:limite], X_tensor[limite:]
y_train, y_test = y_tensor[:limite], y_tensor[limite:]

# =====================================================================
# 3. ARQUITECTURA DE LA RED NEURONAL RECURRENTE (LSTM)
# =====================================================================
class RedNeuronalSolarLSTM(nn.Module):
    def __init__(self, tam_input=1, tam_oculto=64, num_capas=2):
        super(RedNeuronalSolarLSTM, self).__init__()
        self.tam_oculto = tam_oculto
        self.num_capas = num_capas
        self.lstm = nn.LSTM(tam_input, tam_oculto, num_capas, batch_first=True)
        self.linear = nn.Linear(tam_oculto, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_capas, x.size(0), self.tam_oculto).to(x.device)
        c0 = torch.zeros(self.num_capas, x.size(0), self.tam_oculto).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.linear(out[:, -1, :]) # Extraemos el estado latente del último paso temporal
        return out

modelo_lstm = RedNeuronalSolarLSTM()
criterio_perdida = nn.MSELoss()
optimizador = optim.Adam(modelo_lstm.parameters(), lr=0.005)

# =====================================================================
# 4. CICLO DE ENTRENAMIENTO (TRAINING LOOP)
# =====================================================================
print("\n🏋️ Iniciando el entrenamiento de la Red Neuronal PyTorch...")
modelo_lstm.train()
epocas = 30

for epoca in range(epocas):
    optimizador.zero_grad()
    predicciones_train = modelo_lstm(X_train)
    perdida = criterio_perdida(predicciones_train, y_train)
    perdida.backward() # Backpropagation
    optimizador.step() # Optimización de pesos analíticos

    if (epoca + 1) % 5 == 0:
        print(f" 🔄 Época [{epoca+1}/{epocas}] -> Pérdida de Red (MSE): {perdida.item():.6f}")

# =====================================================================
# 5. INFERENCIA Y SERIALIZACIÓN PERSISTENTE (.PKL)
# =====================================================================
print("\n💾 Ejecutando inferencia de validación y serializando artefactos...")
modelo_lstm.eval()
with torch.no_grad():
    predicciones_test_normalizadas = modelo_lstm(X_test).numpy()

# Deshacemos la normalización para almacenar valores reales legibles por el negocio (MW)
reales_mw = escalador.inverse_transform(y_test.numpy())
predicciones_mw = escalador.inverse_transform(predicciones_test_normalizadas)

# Guardamos el estado interno del modelo y los diccionarios de predicciones para el Bloque 2
diccionario_predicciones = {
    'valores_reales_mw': reales_mw.flatten(),
    'valores_predichos_mw': predicciones_mw.flatten(),
    'escalador_transformacion': escalador
}

# Exportación directa a tu ruta de Drive
ruta_pkl_modelo = os.path.join(ruta_drive, 'modelo_pytorch_solar_lstm.pkl')
ruta_pkl_predicciones = os.path.join(ruta_drive, 'predicciones_pytorch_solar.pkl')

joblib.dump(modelo_lstm.state_dict(), ruta_pkl_modelo)
joblib.dump(diccionario_predicciones, ruta_pkl_predicciones)

print(f" -> Modelo estructural guardado en: {ruta_pkl_modelo}")
print(f" -> Predicciones y metadatos exportados en: {ruta_pkl_predicciones}")
print("\n🎉 ¡BLOQUE 1 COMPLETADO! Tus archivos están seguros en Drive. Pasa al Bloque 2.")


🔌 Conectando con Google Drive...
Mounted at /content/drive

⏳ Creando dataset anual de generación solar horaria...
✅ Dataset original guardado exitosamente en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/datos_series_temporales_deep_learning.csv

🏋️ Iniciando el entrenamiento de la Red Neuronal PyTorch...
 🔄 Época [5/30] -> Pérdida de Red (MSE): 0.041806
 🔄 Época [10/30] -> Pérdida de Red (MSE): 0.039705
 🔄 Época [15/30] -> Pérdida de Red (MSE): 0.035354
 🔄 Época [20/30] -> Pérdida de Red (MSE): 0.023933
 🔄 Época [25/30] -> Pérdida de Red (MSE): 0.013515
 🔄 Época [30/30] -> Pérdida de Red (MSE): 0.008826

💾 Ejecutando inferencia de validación y serializando artefactos...
 -> Modelo estructural guardado en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/modelo_pytorch_solar_lstm.pkl
 -> Predicciones y metadatos exportados en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/predicciones_pytorch_solar.pkl

🎉 ¡BLOQUE 1 COMPLE

In [2]:
import os
import joblib
import pandas as pd
import plotly.graph_objects as go
from google.colab import drive

# =====================================================================
# 0. CONFIGURACIÓN Y CARGA DE ARTEFACTOS SERIALIZADOS
# =====================================================================
print("🔌 Conectando con Google Drive para importar la producción de PyTorch...")
drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
ruta_pkl_predicciones = os.path.join(ruta_drive, 'predicciones_pytorch_solar.pkl')

if os.path.exists(ruta_pkl_predicciones):
    # Cargamos el archivo de predicciones e inversions de escala generadas por la red neuronal
    artefactos_produccion = joblib.load(ruta_pkl_predicciones)
    reales_mw = artefactos_produccion['valores_reales_mw']
    predichos_mw = artefactos_produccion['valores_predichos_mw']
    print("✅ Artefactos binarios y arrays de predicciones cargados correctamente.")
else:
    print("❌ Error: No se encontró el archivo de predicciones en tu Drive. Ejecuta el Bloque 1 primero.")

# =====================================================================
# 1. RENDERIZADO DEL PANEL DINÁMICO DE CONTROL OPERATIVO
# =====================================================================
print("\n📊 Construyendo visualización interactiva de control para la mesa de trading...")

# Tomamos una ventana analítica de 120 horas continuas (5 días completos de operación)
muestra_reales = reales_mw[:120]
muestra_predichos = predichos_mw[:120]
eje_tiempo_horas = [f"Hora {i+1}" for i in range(120)]

fig = go.Figure()

# Línea Operativa Real
fig.add_trace(go.Scatter(
    x=eje_tiempo_horas, y=muestra_reales,
    mode='lines+markers', name='Generación Real Sistema (MW)',
    line=dict(color='#1f77b4', width=2.5),
    marker=dict(size=4)
))

# Línea de Predicción de la Red Neuronal LSTM
fig.add_trace(go.Scatter(
    x=eje_tiempo_horas, y=muestra_predichos,
    mode='lines+markers', name='Predicción LSTM (PyTorch Deep Learning)',
    line=dict(color='#ff7f0e', width=2, dash='dash'),
    marker=dict(size=4, symbol='x')
))

fig.update_layout(
    title='🎯 Panel de Control MLOps: Predicción de Energía Solar con Redes Neuronales LSTM',
    xaxis_title='Horizonte de Evaluación Temporal (Ventana de 5 Días)',
    yaxis_title='Potencia de la Central Fotovoltaica (MW)',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

# Imprimir un DataFrame rápido con el resumen numérico de los primeros pasos de inferencia
df_operativo = pd.DataFrame({
    'Real de Red (MW)': reales_mw[:6],
    'Predicción Inteligente (MW)': predichos_mw[:6],
    'Desviación Absoluta (MW)': np.abs(reales_mw[:6] - predichos_mw[:6])
}, index=[f"Paso Hora +{i+1}" for i in range(6)])

print("\n📈 === PRIMEROS PASOS DEL REPORTE DE DESPACHO EN PRODUCCIÓN ===")
print(df_operativo.round(4).to_string())
print("================================================================\n")
print("🔥 Solución de Deep Learning integrada de extremo a extremo con persistencia de archivos.")


🔌 Conectando con Google Drive para importar la producción de PyTorch...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Artefactos binarios y arrays de predicciones cargados correctamente.

📊 Construyendo visualización interactiva de control para la mesa de trading...



📈 === PRIMEROS PASOS DEL REPORTE DE DESPACHO EN PRODUCCIÓN ===
              Real de Red (MW)  Predicción Inteligente (MW)  Desviación Absoluta (MW)
Paso Hora +1            0.0000                    12.261600                   12.2616
Paso Hora +2            0.0000                    13.061300                   13.0613
Paso Hora +3            6.4901                    13.501000                    7.0109
Paso Hora +4           14.5807                    14.903300                    0.3226
Paso Hora +5           19.2470                    17.918100                    1.3290
Paso Hora +6           20.1546                    21.790899                    1.6363

🔥 Solución de Deep Learning integrada de extremo a extremo con persistencia de archivos.


In [3]:
import os
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Flatten, MaxPooling1D
from google.colab import drive
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# =====================================================================
# 0. CONFIGURACIÓN Y MONTAJE DE GOOGLE DRIVE
# =====================================================================
print("🔌 Conectando con Google Drive...")
drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
os.makedirs(ruta_drive, exist_ok=True)
ruta_csv = os.path.join(ruta_drive, 'datos_demanda_deep_learning.csv')

# =====================================================================
# 1. GENERACIÓN DE DATASET DE DEMANDA Y GUARDADO EN CSV
# =====================================================================
print("\n📉 Creando dataset histórico de demanda eléctrica para TensorFlow...")
np.random.seed(24)
fechas = pd.date_range(start="2025-01-01", end="2025-12-31 23:00:00", freq="h")
horas = fechas.hour
dias_semana = fechas.dayofweek

# Simulación de curvas de carga con estacionalidad humana y temperatura
demanda_base = 250 + np.sin((horas - 6) / 24 * 2 * np.pi) * 50 + np.sin((horas - 15) / 24 * 4 * np.pi) * 25
factor_calendario = np.where(dias_semana >= 5, 0.78, 1.0) # Caída los fines de semana
ruido_red = np.random.normal(0, 8, len(fechas))
demanda_mw = np.clip((demanda_base * factor_calendario) + ruido_red, 50, None)

df_demanda = pd.DataFrame({
    'Demanda_MW': demanda_mw,
    'Dia_Semana': dias_semana
}, index=fechas)

df_demanda.to_csv(ruta_csv)
print(f"✅ Dataset guardado exitosamente en: {ruta_csv}")

# =====================================================================
# 2. INGENIERÍA DE TENSORES (VENTANAS DESLIZANTES PARA CNN 1D)
# =====================================================================
escalador = MinMaxScaler(feature_range=(0, 1))
datos_normalizados = escalador.fit_transform(df_demanda[['Demanda_MW']].values)

# Ventana temporal: miramos las últimas 24 horas para predecir la siguiente hora
def crear_ventanas_cnn(datos, ventana_pasada=24):
    X, y = [], []
    for i in range(len(datos) - ventana_pasada):
        X.append(datos[i:(i + ventana_pasada)])
        y.append(datos[i + ventana_pasada])
    return np.array(X), np.array(y)

X_np, y_np = crear_ventanas_cnn(datos_normalizados, ventana_pasada=24)

# Ajustar dimensiones para la capa Conv1D de TensorFlow: [Muestras, Pasos_Temporales, Características]
X_np = X_np.reshape((X_np.shape[0], X_np.shape[1], 1))

# División cronológica estricta para evaluación de producción
limite = int(len(X_np) * 0.8)
X_train, X_test = X_np[:limite], X_np[limite:]
y_train, y_test = y_np[:limite], y_np[limite:]

# Guardar los días de la semana correspondientes al set de pruebas para el análisis posterior del Bloque 2
dias_test = df_demanda['Dia_Semana'].values[24+limite:]

# =====================================================================
# 3. ARQUITECTURA DE LA RED NEURONAL CONVOLUCIONAL 1D (TENSORFLOW)
# =====================================================================
print("\n🏗️ Configurando arquitectura CNN 1D en TensorFlow/Keras...")
modelo_cnn = Sequential([
    # Capa Convolucional que extrae patrones secuenciales horarias
    Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(24, 1)),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(50, activation='relu'),
    Dense(1) # Salida continua lineal para predicción de MW
])

modelo_cnn.compile(optimizer='adam', loss='mse')

# =====================================================================
# 4. ENTRENAMIENTO DE LA RED NEURONAL
# =====================================================================
print("🏋️ Entrenando la Red Convolucional...")
modelo_cnn.fit(X_train, y_train, epochs=15, batch_size=64, verbose=1)

# =====================================================================
# 5. INFERENCIA Y SERIALIZACIÓN PERSISTENTE (.PKL)
# =====================================================================
print("\n💾 Exportando artefactos del modelo a Google Drive...")
predicciones_normalizadas = modelo_cnn.predict(X_test)

# Invertir escalas para regresar a Megavatios (MW) reales de la red eléctrica
reales_mw = escalador.inverse_transform(y_test)
predichos_mw = escalador.inverse_transform(predicciones_normalizadas)

# Diccionario de transferencia de producción
artefactos_demanda = {
    'valores_reales_mw': reales_mw.flatten(),
    'valores_predichos_mw': predichos_mw.flatten(),
    'dias_semana_test': dias_test,
    'escalador': escalador
}

ruta_pkl_modelo_h5 = os.path.join(ruta_drive, 'modelo_tensorflow_demanda_cnn.h5')
ruta_pkl_datos = os.path.join(ruta_drive, 'predicciones_tensorflow_demanda.pkl')

# Guardamos la estructura de pesos de la red y las matrices de inferencia
modelo_cnn.save(ruta_pkl_modelo_h5)
joblib.dump(artefactos_demanda, ruta_pkl_datos)

print(f" -> Arquitectura de red h5 exportada en: {ruta_pkl_modelo_h5}")
print(f" -> Métricas y arrays exportados en: {ruta_pkl_datos}")
print("\n🎉 ¡BLOQUE 1 COMPLETO! Todo listo para ejecutar las gráficas en el Bloque 2.")


🔌 Conectando con Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📉 Creando dataset histórico de demanda eléctrica para TensorFlow...
✅ Dataset guardado exitosamente en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/datos_demanda_deep_learning.csv

🏗️ Configurando arquitectura CNN 1D en TensorFlow/Keras...
🏋️ Entrenando la Red Convolucional...
Epoch 1/15


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0620
Epoch 2/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0143
Epoch 3/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0079
Epoch 4/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0063
Epoch 5/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0057
Epoch 6/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0052
Epoch 7/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0049
Epoch 8/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0049
Epoch 9/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0045
Epoch 10/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0043
Epoch 11/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0042
Epoch 12/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0041
Epoch 13/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0040
Epoch 14/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0038
Epoch 15/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0036


 -> Arquitectura de red h5 exportada en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/modelo_tensorflow_demanda_cnn.h5
 -> Métricas y arrays exportados en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/predicciones_tensorflow_demanda.pkl

🎉 ¡BLOQUE 1 COMPLETO! Todo listo para ejecutar las gráficas en el Bloque 2.


##BLOQUE 2

In [5]:
import os
import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from google.colab import drive

# =====================================================================
# 0. CARGA DE ARTEFACTOS SERIALIZADOS DESDE DRIVE
# =====================================================================
print("🔌 Conectando con Google Drive e importando binarios de TensorFlow...")
drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
ruta_pkl_datos = os.path.join(ruta_drive, 'predicciones_tensorflow_demanda.pkl')

if os.path.exists(ruta_pkl_datos):
    datos_produccion = joblib.load(ruta_pkl_datos)
    reales_mw = datos_produccion['valores_reales_mw']
    predichos_mw = datos_produccion['valores_predichos_mw']
    dias_semana_test = datos_produccion['dias_semana_test']
    print("✅ Arrays de inferencia y metadatos cargados con éxito.")
else:
    print("❌ Error: Ejecuta el Bloque 1 primero para generar los archivos binarios.")

# Calcular el error de predicción (Residuos)
residuos_mw = reales_mw - predichos_mw

# Mapear números de días a texto legible para el análisis corporativo
mapa_dias = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'}
nombres_dias = [mapa_dias[d] for d in dias_semana_test]

# Construcción de un DataFrame maestro para las visualizaciones analíticas de Plotly
df_analisis = pd.DataFrame({
    'Real_MW': reales_mw,
    'Predicho_MW': predichos_mw,
    'Error_MW': residuos_mw,
    'Dia_Texto': nombres_dias,
    'Dia_Num': dias_semana_test
})

# =====================================================================
# 1. GRÁFICA 1: PANEL DE CONTROL DE SERIE TEMPORAL (CORREGIDA)
# =====================================================================
print("\n📊 Generando Gráfica 1: Histórico Operativo con Nombres de Capa Corregidos...")
# Evaluamos una muestra analítica de una semana completa (168 horas continuas)
df_muestra = df_analisis.iloc[:168].copy()
df_muestra['Eje_X'] = [f"Hora +{i+1}" for i in range(168)]

fig1 = go.Figure()

# Traza 1: Demanda Real
fig1.add_trace(go.Scatter(
    x=df_muestra['Eje_X'],
    y=df_muestra['Real_MW'],
    mode='lines',
    name='Demanda Real Sistema',
    line=dict(color='#2ca02c', width=2.5),
    # Guardamos el nombre explícito en la plantilla hover de esta traza
    hovertemplate='<b>Demanda Real Sistema</b><br>Carga de Red: %{y:.4f} MW<extra></extra>'
))

# Traza 2: Predicción CNN
fig1.add_trace(go.Scatter(
    x=df_muestra['Eje_X'],
    y=df_muestra['Predicho_MW'],
    mode='lines',
    name='Predicción CNN 1D',
    line=dict(color='#d62728', width=2, dash='dash'),
    # Guardamos el nombre explícito en la plantilla hover de esta traza
    hovertemplate='<b>Predicción CNN 1D (TensorFlow)</b><br>Carga de Red: %{y:.4f} MW<extra></extra>'
))

# Configuración del contenedor unificado
fig1.update_layout(
    title='🎯 Gráfica 1: Evaluación Semanal de Carga de Red (Real vs Predicción TensorFlow)',
    xaxis_title='Horizonte Temporal (Muestra de 168 Horas Continuas)',
    yaxis_title='Potencia Demandada (MW)',
    template='plotly_white',
    hovermode='x unified' # Une las etiquetas horizontalmente al pasar el cursor
)
fig1.show()

# =====================================================================
# 2. GRÁFICA 2: DISTRIBUCIÓN DE ERRORES CON TOLERANCIA OPERATIVA
# =====================================================================
print("📊 Generando Gráfica 2: Histograma de Residuos...")
fig2 = px.histogram(
    df_analisis, x='Error_MW', nbins=50,
    title='📊 Gráfica 2: Distribución de Errores de Predicción (Residuos de Red Neuronal)',
    labels={'Error_MW': 'Magnitud del Error de Inferencia (Real - Predicho) [MW]'},
    color_discrete_sequence=['#7f7f7f'],
    opacity=0.75
)

# Forzar formato de 4 decimales en el hover del histograma
fig2.update_traces(
    hovertemplate='Rango del Error: %{x:.4f} MW<br>Frecuencia de Ocurrencia: %{y}<extra></extra>'
)

# Líneas de referencia operativas (Línea central en 0 y límites de tolerancia)
fig2.add_shape(type="line", x0=0, y0=0, x1=0, y1=1, xref="x", yref="paper", line=dict(color="black", width=2.5))
fig2.add_shape(type="line", x0=-15, y0=0, x1=-15, y1=1, xref="x", yref="paper", line=dict(color="red", width=1.5, dash="dash"))
fig2.add_shape(type="line", x0=15, y0=0, x1=15, y1=1, xref="x", yref="paper", line=dict(color="red", width=1.5, dash="dash"))

fig2.update_layout(template='plotly_white', yaxis_title='Frecuencia de Horas')
fig2.show()

# =====================================================================
# 3. GRÁFICA 3: DIAGRAMA DE CAJAS (BOXPLOT) DE ERRORES POR DÍA
# =====================================================================
print("📊 Generando Gráfica 3: Boxplot por Día de la Semana...")
df_ordenado = df_analisis.sort_values('Dia_Num')

fig3 = px.box(
    df_ordenado, x='Dia_Texto', y='Error_MW',
    title='📆 Gráfica 3: Análisis de Desviaciones de Inferencia por Día de la Semana',
    labels={'Dia_Texto': 'Días evaluados', 'Error_MW': 'Desviación de Demanda Real - Predicho (MW)'},
    color='Dia_Texto',
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# Configuración del hover con precisión de 4 decimales en las métricas de cajas
fig3.update_traces(
    hovertemplate='Día: %{x}<br>Desviación: %{y:.4f} MW<extra></extra>'
)

fig3.update_layout(template='plotly_white', showlegend=False)
fig3.show()


🔌 Conectando con Google Drive e importando binarios de TensorFlow...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Arrays de inferencia y metadatos cargados con éxito.

📊 Generando Gráfica 1: Histórico Operativo con Nombres de Capa Corregidos...


📊 Generando Gráfica 2: Histograma de Residuos...


📊 Generando Gráfica 3: Boxplot por Día de la Semana...


##BLOQUE DOS VERSION 2

In [6]:
import os
import joblib
import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.graph_objects as go
import plotly.express as px
from google.colab import drive

# =====================================================================
# 0. CARGA DE ARTEFACTOS SERIALIZADOS DESDE DRIVE
# =====================================================================
print("🔌 Conectando con Google Drive e importando binarios de TensorFlow...")
drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
ruta_pkl_datos = os.path.join(ruta_drive, 'predicciones_tensorflow_demanda.pkl')

if os.path.exists(ruta_pkl_datos):
    datos_produccion = joblib.load(ruta_pkl_datos)
    reales_mw = datos_produccion['valores_reales_mw']
    predichos_mw = datos_produccion['valores_predichos_mw']
    dias_semana_test = datos_produccion['dias_semana_test']
    print("✅ Arrays de inferencia y metadatos cargados con éxito.")
else:
    print("❌ Error: Ejecuta el Bloque 1 primero para generar los archivos binarios.")

# Calcular el error de predicción (Residuos)
residuos_mw = reales_mw - predichos_mw

# Mapear números de días a texto legible para el análisis corporativo
mapa_dias = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'}
nombres_dias = [mapa_dias[d] for d in dias_semana_test]

# Construcción de un DataFrame maestro para las visualizaciones analíticas de Plotly
df_analisis = pd.DataFrame({
    'Real_MW': reales_mw,
    'Predicho_MW': predichos_mw,
    'Error_MW': residuos_mw,
    'Dia_Texto': nombres_dias,
    'Dia_Num': dias_semana_test
})

# =====================================================================
# 1. GRÁFICA 1: PANEL DE CONTROL DE SERIE TEMPORAL
# =====================================================================
print("\n📊 Generando Gráfica 1: Histórico Operativo...")
df_muestra = df_analisis.iloc[:168].copy()
df_muestra['Eje_X'] = [f"Hora +{i+1}" for i in range(168)]

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=df_muestra['Eje_X'], y=df_muestra['Real_MW'],
    mode='lines', name='Demanda Real Sistema',
    line=dict(color='#2ca02c', width=2.5),
    hovertemplate='<b>Demanda Real Sistema</b><br>Carga de Red: %{y:.4f} MW<extra></extra>'
))
fig1.add_trace(go.Scatter(
    x=df_muestra['Eje_X'], y=df_muestra['Predicho_MW'],
    mode='lines', name='Predicción CNN 1D',
    line=dict(color='#d62728', width=2, dash='dash'),
    hovertemplate='<b>Predicción CNN 1D (TensorFlow)</b><br>Carga de Red: %{y:.4f} MW<extra></extra>'
))
fig1.update_layout(
    title='🎯 Gráfica 1: Evaluación Semanal de Carga de Red (Real vs Predicción TensorFlow)',
    xaxis_title='Horizonte Temporal (Muestra de 168 Horas Continuas)',
    yaxis_title='Potencia Demandada (MW)',
    template='plotly_white',
    hovermode='x unified'
)
fig1.show()

# =====================================================================
# 2. GRÁFICA 2: DISTRIBUCIÓN DE ERRORES (BARRAS AZUL CLARO + CURVA DE GAUSS)
# =====================================================================
print("📊 Generando Gráfica 2: Histograma de Residuos con Curva de Gauss...")

# Estadísticos muestrales de los errores reales para trazar la campana teórica
media_error = np.mean(residuos_mw)
desviacion_error = np.std(residuos_mw)

# Generar el eje X continuo para la curva matemática de Gauss
x_gauss = np.linspace(min(residuos_mw), max(residuos_mw), 200)
# Calcular la densidad de probabilidad matemática (Campana de Gauss teórica)
y_gauss = stats.norm.pdf(x_gauss, media_error, desviacion_error)

fig2 = go.Figure()

# Traza del Histograma con densidad de probabilidad (histnorm='probability density')
fig2.add_trace(go.Histogram(
    x=df_analisis['Error_MW'],
    nbinsx=50,
    histnorm='probability density',
    name='Frecuencia del Error',
    marker_color='#a6c8e0', # Color azul claro profesional (soft blue)
    opacity=0.85,
    hovertemplate='Rango del Error: %{x:.4f} MW<br>Densidad: %{y:.4f}<extra></extra>'
))

# Traza de la Curva de Gauss teórica superpuesta
fig2.add_trace(go.Scatter(
    x=x_gauss,
    y=y_gauss,
    mode='lines',
    name='Curva de Gauss Teórica',
    line=dict(color='#1f77b4', width=3), # Azul oscuro para contrastar con las barras
    hovertemplate='Punto del Error: %{x:.4f} MW<br>Densidad Teórica: %{y:.4f}<extra></extra>'
))

# Determinar la altura máxima aproximada del gráfico para dibujar correctamente las líneas verticales
alto_maximo_y = max(y_gauss) * 1.15

# Líneas de referencia operativas (Línea central en 0 y límites de tolerancia industriales)
fig2.add_shape(type="line", x0=0, y0=0, x1=0, y1=alto_maximo_y, line=dict(color="black", width=2.5))
fig2.add_shape(type="line", x0=-15, y0=0, x1=-15, y1=alto_maximo_y, line=dict(color="red", width=1.5, dash="dash"))
fig2.add_shape(type="line", x0=15, y0=0, x1=15, y1=alto_maximo_y, line=dict(color="red", width=1.5, dash="dash"))

fig2.update_layout(
    title=f'📊 Gráfica 2: Distribución de Errores vs. Campana de Gauss Teórica (μ={media_error:.2f}, σ={desviacion_error:.2f})',
    xaxis_title='Magnitud del Error de Inferencia (Real - Predicho) [MW]',
    yaxis_title='Densidad de Probabilidad',
    template='plotly_white',
    hovermode='closest'
)
fig2.show()

# =====================================================================
# 3. GRÁFICA 3: DIAGRAMA DE CAJAS (BOXPLOT) DE ERRORES POR DÍA
# =====================================================================
print("📊 Generando Gráfica 3: Boxplot por Día de la Semana...")
df_ordenado = df_analisis.sort_values('Dia_Num')

fig3 = px.box(
    df_ordenado, x='Dia_Texto', y='Error_MW',
    title='📆 Gráfica 3: Análisis de Desviaciones de Inferencia por Día de la Semana',
    labels={'Dia_Texto': 'Días evaluados', 'Error_MW': 'Desviación de Demanda Real - Predicho (MW)'},
    color='Dia_Texto',
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig3.update_traces(
    hovertemplate='Día: %{x}<br>Desviación: %{y:.4f} MW<extra></extra>'
)
fig3.update_layout(template='plotly_white', showlegend=False)
fig3.show()


🔌 Conectando con Google Drive e importando binarios de TensorFlow...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Arrays de inferencia y metadatos cargados con éxito.

📊 Generando Gráfica 1: Histórico Operativo...


📊 Generando Gráfica 2: Histograma de Residuos con Curva de Gauss...


📊 Generando Gráfica 3: Boxplot por Día de la Semana...


In [7]:
import os
import joblib
import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.graph_objects as go
import plotly.express as px
from google.colab import drive
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =====================================================================
# 0. CARGA DE ARTEFACTOS SERIALIZADOS DESDE DRIVE
# =====================================================================
print("🔌 Conectando con Google Drive e importando binarios de TensorFlow...")
drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
ruta_pkl_datos = os.path.join(ruta_drive, 'predicciones_tensorflow_demanda.pkl')

if os.path.exists(ruta_pkl_datos):
    datos_produccion = joblib.load(ruta_pkl_datos)
    reales_mw = datos_produccion['valores_reales_mw']
    predichos_mw = datos_produccion['valores_predichos_mw']
    dias_semana_test = datos_produccion['dias_semana_test']
    print("✅ Arrays de inferencia y metadatos cargados con éxito.")
else:
    print("❌ Error: Ejecuta el Bloque 1 primero para generar los archivos binarios.")

# Calcular el error de predicción (Residuos)
residuos_mw = reales_mw - predichos_mw

# Mapear números de días a texto legible para el análisis corporativo
mapa_dias = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'}
nombres_dias = [mapa_dias[d] for d in dias_semana_test]

# Construcción de un DataFrame maestro para las visualizaciones analíticas de Plotly
df_analisis = pd.DataFrame({
    'Real_MW': reales_mw,
    'Predicho_MW': predichos_mw,
    'Error_MW': residuos_mw,
    'Dia_Texto': nombres_dias,
    'Dia_Num': dias_semana_test
})

# =====================================================================
# 1. GRÁFICA 1: PANEL DE CONTROL DE SERIE TEMPORAL
# =====================================================================
print("\n📊 Generando Gráfica 1: Histórico Operativo...")
df_muestra = df_analisis.iloc[:168].copy()
df_muestra['Eje_X'] = [f"Hora +{i+1}" for i in range(168)]

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=df_muestra['Eje_X'], y=df_muestra['Real_MW'],
    mode='lines', name='Demanda Real Sistema',
    line=dict(color='#2ca02c', width=2.5),
    hovertemplate='<b>Demanda Real Sistema</b><br>Carga de Red: %{y:.4f} MW<extra></extra>'
))
fig1.add_trace(go.Scatter(
    x=df_muestra['Eje_X'], y=df_muestra['Predicho_MW'],
    mode='lines', name='Predicción CNN 1D',
    line=dict(color='#d62728', width=2, dash='dash'),
    hovertemplate='<b>Predicción CNN 1D (TensorFlow)</b><br>Carga de Red: %{y:.4f} MW<extra></extra>'
))
fig1.update_layout(
    title='🎯 Gráfica 1: Evaluación Semanal de Carga de Red (Real vs Predicción TensorFlow)',
    xaxis_title='Horizonte Temporal (Muestra de 168 Horas Continuas)',
    yaxis_title='Potencia Demandada (MW)',
    template='plotly_white',
    hovermode='x unified'
)
fig1.show()

# =====================================================================
# 2. GRÁFICA 2: DISTRIBUCIÓN DE ERRORES (BARRAS AZUL CLARO + CURVA DE GAUSS)
# =====================================================================
print("📊 Generando Gráfica 2: Histograma de Residuos con Curva de Gauss...")

media_error = np.mean(residuos_mw)
desviacion_error = np.std(residuos_mw)

x_gauss = np.linspace(min(residuos_mw), max(residuos_mw), 200)
y_gauss = stats.norm.pdf(x_gauss, media_error, desviacion_error)

fig2 = go.Figure()
fig2.add_trace(go.Histogram(
    x=df_analisis['Error_MW'],
    nbinsx=50,
    histnorm='probability density',
    name='Frecuencia del Error',
    marker_color='#a6c8e0',
    opacity=0.85,
    hovertemplate='Rango del Error: %{x:.4f} MW<br>Densidad: %{y:.4f}<extra></extra>'
))
fig2.add_trace(go.Scatter(
    x=x_gauss,
    y=y_gauss,
    mode='lines',
    name='Curva de Gauss Teórica',
    line=dict(color='#1f77b4', width=3),
    hovertemplate='Punto del Error: %{x:.4f} MW<br>Densidad Teórica: %{y:.4f}<extra></extra>'
))

alto_maximo_y = max(y_gauss) * 1.15
fig2.add_shape(type="line", x0=0, y0=0, x1=0, y1=alto_maximo_y, line=dict(color="black", width=2.5))
fig2.add_shape(type="line", x0=-15, y0=0, x1=-15, y1=alto_maximo_y, line=dict(color="red", width=1.5, dash="dash"))
fig2.add_shape(type="line", x0=15, y0=0, x1=15, y1=alto_maximo_y, line=dict(color="red", width=1.5, dash="dash"))

fig2.update_layout(
    title=f'📊 Gráfica 2: Distribución de Errores vs. Campana de Gauss Teórica (μ={media_error:.2f}, σ={desviacion_error:.2f})',
    xaxis_title='Magnitud del Error de Inferencia (Real - Predicho) [MW]',
    yaxis_title='Densidad de Probabilidad',
    template='plotly_white',
    hovermode='closest'
)
fig2.show()

# =====================================================================
# 3. GRÁFICA 3: DIAGRAMA DE CAJAS (BOXPLOT) DE ERRORES POR DÍA
# =====================================================================
print("📊 Generando Gráfica 3: Boxplot por Día de la Semana...")
df_ordenado = df_analisis.sort_values('Dia_Num')

fig3 = px.box(
    df_ordenado, x='Dia_Texto', y='Error_MW',
    title='📆 Gráfica 3: Análisis de Desviaciones de Inferencia por Día de la Semana',
    labels={'Dia_Texto': 'Días evaluados', 'Error_MW': 'Desviación de Demanda Real - Predicho (MW)'},
    color='Dia_Texto',
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig3.update_traces(
    hovertemplate='Día: %{x}<br>Desviación: %{y:.4f} MW<extra></extra>'
)
fig3.update_layout(template='plotly_white', showlegend=False)
fig3.show()

# =====================================================================
# 4. NUEVA SECCIÓN: EVALUACIÓN DE MÉTRICAS MATEMÁTICAS GLOBALES (.4f)
# =====================================================================
print("\n🧮 Calculando auditoría de métricas de rendimiento globales...")

# Cómputo matemático formal del rendimiento de la red neuronal
mae_global = mean_absolute_error(reales_mw, predichos_mw)
mse_global = mean_squared_error(reales_mw, predichos_mw)
rmse_global = np.sqrt(mse_global)
r2_global = r2_score(reales_mw, predichos_mw)

# Impresión por consola formateada con formato estricto de 4 decimales
print("\n================ METRICAS MATEMATICAS GLOBALES (MLES) ================")
print(f" Error Absoluto Medio (MAE):               {mae_global:.4f} MW")
print(f" Error Cuadrático Medio (MSE):             {mse_global:.4f} MW²")
print(f" Raíz del Error Cuadrático Medio (RMSE):   {rmse_global:.4f} MW")
print(f" Coeficiente de Determinación (R² Score):   {r2_global:.4f}")
print("======================================================================")


🔌 Conectando con Google Drive e importando binarios de TensorFlow...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Arrays de inferencia y metadatos cargados con éxito.

📊 Generando Gráfica 1: Histórico Operativo...


📊 Generando Gráfica 2: Histograma de Residuos con Curva de Gauss...


📊 Generando Gráfica 3: Boxplot por Día de la Semana...



🧮 Calculando auditoría de métricas de rendimiento globales...

================ METRICAS MATEMATICAS GLOBALES (MLES) ================
 Error Absoluto Medio (MAE):               9.1626 MW
 Error Cuadrático Medio (MSE):             143.9021 MW²
 Raíz del Error Cuadrático Medio (RMSE):   11.9959 MW
 Coeficiente de Determinación (R² Score):   0.9298


##Modelización de Precios de Mercado Pool

##BLOQUE 1

In [8]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from google.colab import drive
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# =====================================================================
# 0. CONFIGURACIÓN Y MONTAJE DE GOOGLE DRIVE
# =====================================================================
print("🔌 Conectando con Google Drive...")
drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
os.makedirs(ruta_drive, exist_ok=True)
ruta_csv = os.path.join(ruta_drive, 'datos_precios_mercado_deep_learning.csv')

# =====================================================================
# 1. GENERACIÓN DE DATASET FINANCIERO Y GUARDADO EN CSV
# =====================================================================
print("\n💶 Creando dataset de precios de mercado pool para PyTorch...")
np.random.seed(99)
fechas = pd.date_range(start="2026-06-01", end="2026-06-30", freq="h")
horas = fechas.hour
dias_semana = fechas.dayofweek

# Simulación acoplada de oferta, demanda y commodities
demanda = 220 + np.sin((horas - 7) / 24 * 2 * np.pi) * 50 + np.where(dias_semana >= 5, -40, 0) + np.random.normal(0, 5, len(fechas))
solar = np.where((horas > 5) & (horas < 20), np.sin((horas - 5) / 14 * np.pi) * 120 + np.random.normal(0, 5, len(fechas)), 0)
solar = np.clip(solar, 0, None)
margen_reserva = demanda - solar

precio_gas = 35 + np.cumsum(np.random.normal(0, 0.4, len(fechas)))
precio_co2 = 70 + np.cumsum(np.random.normal(0, 0.2, len(fechas)))

# Fórmula estructural del mercado marginalista con picos de precios
precio_base_gas = (precio_gas * 2) + (precio_co2 * 0.4)
componente_escasez = np.where(margen_reserva > 180, ((margen_reserva - 180) ** 1.8) * 1.5, 0)
componente_solar = np.where((solar > 80) & (demanda < 200), -40, 0)
precio_pool = np.clip(precio_base_gas + componente_escasez + componente_solar + np.random.normal(0, 4, len(fechas)), 0, None)

df_precios = pd.DataFrame({
    'Demanda_MW': demanda,
    'Generacion_Solar_MW': solar,
    'Margen_Reserva_MW': margen_reserva,
    'Precio_Gas_Euros_MWh': precio_gas,
    'Precio_CO2_Euros_Ton': precio_co2,
    'Precio_Pool_Euros_MWh': precio_pool,
    'Dia_Semana': dias_semana
}, index=fechas)

df_precios.to_csv(ruta_csv)
print(f"✅ Dataset financiero guardado exitosamente en: {ruta_csv}")

# =====================================================================
# 2. INGENIERÍA DE VARIABLES Y PREPARACIÓN DE TENSORES
# =====================================================================
# Añadimos lags financieros antes de escalar para dar memoria de trading al modelo
df_precios['Precio_Lag1'] = df_precios['Precio_Pool_Euros_MWh'].shift(1)
df_precios['Precio_Lag2'] = df_precios['Precio_Pool_Euros_MWh'].shift(2)
df_precios['Precio_Lag24'] = df_precios['Precio_Pool_Euros_MWh'].shift(24)
df_precios.dropna(inplace=True)

# Listado oficial de características predictoras
features = ['Demanda_MW', 'Generacion_Solar_MW', 'Margen_Reserva_MW',
            'Precio_Gas_Euros_MWh', 'Precio_CO2_Euros_Ton', 'Precio_Lag1', 'Precio_Lag2', 'Precio_Lag24']

X_raw = df_precios[features].values
y_raw = df_precios[['Precio_Pool_Euros_MWh']].values

# Normalización individual para aislar correctamente la variable objetivo
escalador_X = MinMaxScaler(feature_range=(0, 1))
escalador_y = MinMaxScaler(feature_range=(0, 1))

X_scaled = escalador_X.fit_transform(X_raw)
y_scaled = escalador_y.fit_transform(y_raw)

# División cronológica del set de datos comercial (80% entrenamiento, 20% testeo)
limite = int(len(X_scaled) * 0.8)
X_train, X_test = X_scaled[:limite], X_scaled[limite:]
y_train, y_test = y_scaled[:limite], y_scaled[limite:]

# Guardar metadatos temporales de testeo
dias_test = df_precios['Dia_Semana'].values[limite:]

# Transformación formal a Tensores de PyTorch
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

# =====================================================================
# 3. ARQUITECTURA DE LA RED NEURONAL PROFUNDA (MLP CON DROPOUT)
# =====================================================================
class RedProfundaPrecios(nn.Module):
    def __init__(self, input_dim=8):
        super(RedProfundaPrecios, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.15), # Previene el sobreajuste ante ruido financiero
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1) # Predicción del valor continuo del pool
        )

    def forward(self, x):
        return self.network(x)

modelo_precios = RedProfundaPrecios()
criterio_perdida = nn.MSELoss()
optimizador = optim.Adam(modelo_precios.parameters(), lr=0.003)

# =====================================================================
# 4. CICLO DE ENTRENAMIENTO (TRAINING LOOP)
# =====================================================================
print("\n🏋️ Entrenando la Red Neuronal PyTorch para Precios Pool...")
modelo_precios.train()
epocas = 50

for epoca in range(epocas):
    optimizador.zero_grad()
    predicciones_train = modelo_precios(X_train_t)
    perdida = criterio_perdida(predicciones_train, y_train_t)
    perdida.backward()
    optimizador.step()

    if (epoca + 1) % 10 == 0:
        print(f" 🔄 Época [{epoca+1}/{epocas}] -> Pérdida del Mercado (MSE): {perdida.item():.6f}")

# =====================================================================
# 5. INFERENCIA Y SERIALIZACIÓN PERSISTENTE (.PKL)
# =====================================================================
print("\n💾 Exportando artefactos del modelo de Precios a Google Drive...")
modelo_precios.eval()
with torch.no_grad():
    predicciones_normalizadas = modelo_precios(X_test_t).numpy()

# Deshacemos la normalización para recuperar los Euros por Megavatio-Hora (€/MWh) reales
reales_euros = escalador_y.inverse_transform(y_test)
predichos_euros = escalador_y.inverse_transform(predicciones_normalizadas)

# Estructuración de artefactos finales
artefactos_precios = {
    'valores_reales_mw': reales_euros.flatten(),
    'valores_predichos_mw': predichos_euros.flatten(),
    'dias_semana_test': dias_test,
    'escalador_y': escalador_y
}

ruta_pkl_modelo = os.path.join(ruta_drive, 'modelo_pytorch_precios_mlp.pkl')
ruta_pkl_datos = os.path.join(ruta_drive, 'predicciones_pytorch_precios.pkl')

torch.save(modelo_precios.state_dict(), ruta_pkl_modelo)
joblib.dump(artefactos_precios, ruta_pkl_datos)

print(f" -> Pesos de red PyTorch exportados en: {ruta_pkl_modelo}")
print(f" -> Métricas y arrays exportados en: {ruta_pkl_datos}")
print("\n🎉 ¡BLOQUE 1 COMPLETO! Todo listo para ejecutar las gráficas en el Bloque 2.")


🔌 Conectando con Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

💶 Creando dataset de precios de mercado pool para PyTorch...
✅ Dataset financiero guardado exitosamente en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/datos_precios_mercado_deep_learning.csv

🏋️ Entrenando la Red Neuronal PyTorch para Precios Pool...
 🔄 Época [10/50] -> Pérdida del Mercado (MSE): 0.012139
 🔄 Época [20/50] -> Pérdida del Mercado (MSE): 0.007006
 🔄 Época [30/50] -> Pérdida del Mercado (MSE): 0.004668
 🔄 Época [40/50] -> Pérdida del Mercado (MSE): 0.004257
 🔄 Época [50/50] -> Pérdida del Mercado (MSE): 0.002774

💾 Exportando artefactos del modelo de Precios a Google Drive...
 -> Pesos de red PyTorch exportados en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/modelo_pytorch_precios_mlp.pkl
 -> Métricas y arrays exportados en: /content/drive/MyDrive/COLAB_NOTEBOOKS/M

##BLOQUE 2

In [9]:
import os
import joblib
import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.graph_objects as go
import plotly.express as px
from google.colab import drive
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =====================================================================
# 0. CARGA DE ARTEFACTOS SERIALIZADOS DESDE DRIVE
# =====================================================================
print("🔌 Conectando con Google Drive e importando binarios de PyTorch...")
drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
ruta_pkl_datos = os.path.join(ruta_drive, 'predicciones_pytorch_precios.pkl')

if os.path.exists(ruta_pkl_datos):
    datos_produccion = joblib.load(ruta_pkl_datos)
    reales_mw = datos_produccion['valores_reales_mw']
    predichos_mw = datos_produccion['valores_predichos_mw']
    dias_semana_test = datos_produccion['dias_semana_test']
    print("✅ Arrays de inferencia y metadatos financieros cargados con éxito.")
else:
    print("❌ Error: Ejecuta el Bloque 1 primero para generar los archivos binarios.")

# Calcular el error de predicción (Residuos)
residuos_mw = reales_mw - predichos_mw

# Mapear números de días a texto legible para la mesa de trading
mapa_dias = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'}
nombres_dias = [mapa_dias[d] for d in dias_semana_test]

# Construcción de un DataFrame maestro para las visualizaciones analíticas de Plotly
df_analisis = pd.DataFrame({
    'Real_MW': reales_mw,
    'Predicho_MW': predichos_mw,
    'Error_MW': residuos_mw,
    'Dia_Texto': nombres_dias,
    'Dia_Num': dias_semana_test
})

# =====================================================================
# 1. GRÁFICA 1: PANEL DE CONTROL DE SERIE TEMPORAL (REAL VS PREDICHO)
# =====================================================================
print("\n📊 Generando Gráfica 1: Histórico Operativo del Pool...")
# Evaluamos una muestra analítica de 5 días de trading (120 horas continuas)
df_muestra = df_analisis.iloc[:120].copy()
df_muestra['Eje_X'] = [f"Hora +{i+1}" for i in range(120)]

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=df_muestra['Eje_X'], y=df_muestra['Real_MW'],
    mode='lines', name='Precio Real Pool',
    line=dict(color='#bcbd22', width=2.5),
    hovertemplate='<b>Precio Real Pool</b><br>Valor Mercado: %{y:.4f} €/MWh<extra></extra>'
))
fig1.add_trace(go.Scatter(
    x=df_muestra['Eje_X'], y=df_muestra['Predicho_MW'],
    mode='lines', name='Predicción PyTorch MLP',
    line=dict(color='#17becf', width=2, dash='dash'),
    hovertemplate='<b>Predicción PyTorch MLP</b><br>Valor Mercado: %{y:.4f} €/MWh<extra></extra>'
))
fig1.update_layout(
    title='🎯 Gráfica 1: Evaluación Comercial de Precios Pool (€/MWh) (Real vs Predicción PyTorch Deep Learning)',
    xaxis_title='Horizonte Temporal de Trading (Muestra de 120 Horas Continuas)',
    yaxis_title='Precio del Mercado Mayorista (€/MWh)',
    template='plotly_white',
    hovermode='x unified'
)
fig1.show()

# =====================================================================
# 2. GRÁFICA 2: DISTRIBUCIÓN DE ERRORES (BARRAS AZUL CLARO + CURVA DE GAUSS)
# =====================================================================
print("📊 Generando Gráfica 2: Histograma de Residuos con Curva de Gauss...")

media_error = np.mean(residuos_mw)
desviacion_error = np.std(residuos_mw)

x_gauss = np.linspace(min(residuos_mw), max(residuos_mw), 200)
y_gauss = stats.norm.pdf(x_gauss, media_error, desviacion_error)

fig2 = go.Figure()
fig2.add_trace(go.Histogram(
    x=df_analisis['Error_MW'],
    nbinsx=50,
    histnorm='probability density',
    name='Frecuencia del Error',
    marker_color='#a6c8e0', # Azul claro solicitado
    opacity=0.85,
    hovertemplate='Rango del Error: %{x:.4f} €/MWh<br>Densidad: %{y:.4f}<extra></extra>'
))
fig2.add_trace(go.Scatter(
    x=x_gauss,
    y=y_gauss,
    mode='lines',
    name='Curva de Gauss Teórica',
    line=dict(color='#1f77b4', width=3),
    hovertemplate='Punto del Error: %{x:.4f} €/MWh<br>Densidad Teórica: %{y:.4f}<extra></extra>'
))

alto_maximo_y = max(y_gauss) * 1.15
fig2.add_shape(type="line", x0=0, y0=0, x1=0, y1=alto_maximo_y, line=dict(color="black", width=2.5))
# Tolerancia de desviación financiera en el mercado español/europeo habitual (ej. ±10 €/MWh)
fig2.add_shape(type="line", x0=-10, y0=0, x1=-10, y1=alto_maximo_y, line=dict(color="red", width=1.5, dash="dash"))
fig2.add_shape(type="line", x0=10, y0=0, x1=10, y1=alto_maximo_y, line=dict(color="red", width=1.5, dash="dash"))

fig2.update_layout(
    title=f'📊 Gráfica 2: Distribución de Errores Financieros vs. Campana de Gauss Teórica (μ={media_error:.2f}, σ={desviacion_error:.2f})',
    xaxis_title='Magnitud del Error de Precio (Real - Predicho) [€/MWh]',
    yaxis_title='Densidad de Probabilidad',
    template='plotly_white',
    hovermode='closest'
)
fig2.show()

# =====================================================================
# 3. GRÁFICA 3: DIAGRAMA DE CAJAS (BOXPLOT) DE ERRORES POR DÍA
# =====================================================================
print("📊 Generando Gráfica 3: Boxplot por Día de la Semana...")
df_ordenado = df_analisis.sort_values('Dia_Num')

fig3 = px.box(
    df_ordenado, x='Dia_Texto', y='Error_MW',
    title='📆 Gráfica 3: Análisis de Desviaciones de Precio por Día de la Semana',
    labels={'Dia_Texto': 'Días de Operación Financiera', 'Error_MW': 'Desviación de Precio Real - Predicho (€/MWh)'},
    color='Dia_Texto',
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig3.update_traces(
    hovertemplate='Día: %{x}<br>Desviación: %{y:.4f} €/MWh<extra></extra>'
)
fig3.update_layout(template='plotly_white', showlegend=False)
fig3.show()

# =====================================================================
# 4. SECCIÓN DE EVALUACIÓN DE MÉTRICAS MATEMÁTICAS GLOBALES (.4f)
# =====================================================================
print("\n🧮 Calculando auditoría de métricas de rendimiento globales para Precios Pool...")

mae_global = mean_absolute_error(reales_mw, predichos_mw)
mse_global = mean_squared_error(reales_mw, predichos_mw)
rmse_global = np.sqrt(mse_global)
r2_global = r2_score(reales_mw, predichos_mw)

print("\n================ METRICAS MATEMATICAS GLOBALES (MLES - PRECIOS) ================")
print(f" Error Absoluto Medio (MAE):               {mae_global:.4f} €/MWh")
print(f" Error Cuadrático Medio (MSE):             {mse_global:.4f} (€/MWh)²")
print(f" Raíz del Error Cuadrático Medio (RMSE):   {rmse_global:.4f} €/MWh")
print(f" Coeficiente de Determinación (R² Score):   {r2_global:.4f}")
print("=================================================================================")


🔌 Conectando con Google Drive e importando binarios de PyTorch...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Arrays de inferencia y metadatos financieros cargados con éxito.

📊 Generando Gráfica 1: Histórico Operativo del Pool...


📊 Generando Gráfica 2: Histograma de Residuos con Curva de Gauss...


📊 Generando Gráfica 3: Boxplot por Día de la Semana...



🧮 Calculando auditoría de métricas de rendimiento globales para Precios Pool...

================ METRICAS MATEMATICAS GLOBALES (MLES - PRECIOS) ================
 Error Absoluto Medio (MAE):               39.5387 €/MWh
 Error Cuadrático Medio (MSE):             7490.0070 (€/MWh)²
 Raíz del Error Cuadrático Medio (RMSE):   86.5448 €/MWh
 Coeficiente de Determinación (R² Score):   0.8634


In [10]:
import os
from google.colab import drive

# 1. Conectar a Google Drive
print("🔌 Conectando con Google Drive para actualizar documentación...")
drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
os.makedirs(ruta_drive, exist_ok=True)
ruta_readme = os.path.join(ruta_drive, 'README.md')

# 2. Texto estructurado en Markdown técnico para reclutadores
contenido_readme = """# Ecosistema de Modelos de Deep Learning Aplicados al Sector Eléctrico ⚡🧠

Este repositorio contiene un entorno avanzado de producción, entrenamiento y validación de modelos predictivos de aprendizaje profundo (**Deep Learning**) implementados en **PyTorch** y **TensorFlow/Keras**. El ecosistema está diseñado específicamente para abordar los desafíos operativos y de mercado más exigentes que enfrentan las *utilities* y los operadores de sistemas eléctricos modernos en el marco de la transición energética.

---

## 📁 Estructura del Almacenamiento en Producción

El proyecto está diseñado bajo principios de desacoplamiento de software y prácticas robustas de *MLOps*. Todos los datos históricos estructurados, arquitecturas de redes y binarios de inferencia se sincronizan automáticamente en la siguiente ruta de producción:

```text
/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/
├── datos_series_temporales_deep_learning.csv   # Dataset anual del Punto 1 (Solar)
├── datos_demanda_deep_learning.csv             # Dataset anual del Punto 2 (Demanda)
├── datos_precios_mercado_deep_learning.csv     # Dataset mensual del Punto 3 (Precios)
├── modelo_pytorch_solar_lstm.pkl               # Pesos de la Red LSTM (PyTorch) - Punto 1
├── predicciones_pytorch_solar.pkl              # Métricas e inferencias del Punto 1
├── modelo_tensorflow_demanda_cnn.h5            # Arquitectura y pesos CNN 1D (TF) - Punto 2
├── predicciones_tensorflow_demanda.pkl         # Métricas e inferencias del Punto 2
├── modelo_pytorch_precios_mlp.pkl              # Pesos de la Red MLP (PyTorch) - Punto 3
└── predicciones_pytorch_precios.pkl            # Métricas e inferencias del Punto 3
```

---

## 🏗️ Desglose Técnico de Soluciones Industriales

### ☀️ 1. Pronóstico de Generación Renovable (Redes LSTM en PyTorch)
*   **Caso de Uso Real:** Predicción de curvas horarias de potencia fotovoltaica (MW) para mitigar el impacto financiero de los desbalances energéticos y optimizar las ofertas de venta en subastas intradiarias.
*   **Arquitectura:** Red Neuronal Recurrente con celdas de **Memoria a Largo Corto Plazo (LSTM)** de dos capas acopladas, diseñada para capturar la memoria térmica, los frentes nubosos y las rampas de irradiancia.
*   **Ingeniería de Datos:** Transformación de la serie temporal mediante técnicas de ventana deslizante continua de 24 horas hacia el pasado, procesada mediante normalización de escala min-max.

### 📉 2. Predicción de Demanda Eléctrica (Redes Convolucionales 1D en TensorFlow)
*   **Caso de Uso Real:** Auditoría de carga del sistema para que la mesa de despacho pueda programar el encendido eficiente del parque térmico convencional, previniendo sobrecargas en subestaciones y apagones estructurales.
*   **Arquitectura:** Red Neuronal Convolucional Unidimensional (**CNN 1D**) acoplada a capas de reducción espacial (`MaxPooling1D`) y capas densas profundas en TensorFlow/Keras. Funciona extrayendo micro-patrones locales de consumo.

### 💶 3. Modelización de Precios de Mercado Pool (MLP Profunda en PyTorch)
*   **Caso de Uso Real:** Pronóstico de la volatilidad del precio marginal del pool eléctrico (€/MWh) y anticipación de picos de escasez (*Price Spikes*) para mitigar riesgos financieros en la mesa de trading de energía.
*   **Arquitectura:** Red Neuronal Profunda Multicapa (**Perceptrón Multicapa - MLP**) en PyTorch con capas densas secuenciales y regularización activa mediante **Dropout (15%)** para filtrar el ruido y anomalías propias del mercado de subastas.
*   **Inyección de Variables Financieras:** Inclusión de curvas de costes de *commodities* internacionales (Gas y CO₂) acopladas a lags estructurales de mercado (`Precio_Lag1`, `Precio_Lag24`).

---

## 📊 Paneles Analíticos Interactivos (Plotly MLOps)

El proyecto destaca por su madurez visual, sustituyendo gráficos estáticos convencionales por cuadros de mando dinámicos e interactivos en **Plotly (HTML)**. Los paneles operativos cuentan con las siguientes especificaciones técnicas:

1.  **Panel de Serie Temporal (Gráfica 1):** Comparativa directa de curvas reales vs. predicciones del modelo con un sistema de *hover* unificado en el eje X (`hovermode='x unified'`) y nombres fijos de capas para auditorías precisas de despacho.
2.  **Análisis de Residuos de Gauss (Gráfica 2):** Histograma de errores reales con color personalizado azul claro (`#a6c8e0`), superpuesto matemáticamente con una **Campana de Gauss Teórica** continua calculada dinámicamente mediante densidad de probabilidad. Incluye líneas de tolerancia industriales para identificar sesgos financieros.
3.  **Auditoría Temporal Boxplot (Gráfica 3):** Diagramas de cajas interactivos distribuidos por día de la semana para evaluar la dispersión de las desviaciones de la inteligencia artificial entre días laborales y el parón industrial del fin de semana.

---

## 🧮 Auditoría y Precisión Estricta de Métricas (.4f)

Tanto las herramientas de información flotante (*hover*) de los gráficos de Plotly como la consola corporativa de auditorías ejecutan formatos estrictos de redondeo de **4 cifras decimales (`.4f`)** para las métricas globales del negocio:
*   **MAE** (Error Absoluto Medio)
*   **MSE** (Error Cuadrático Medio)
*   **RMSE** (Raíz del Error Cuadrático Medio)
*   **R² Score** (Coeficiente de Determinación)

Garantizando una trazabilidad matemática exacta, libre de truncamientos imprecisos.

---
**Desarrollado bajo rigurosos criterios de Machine Learning Engineering para el sector utilities.** 🚀
"""

# 3. Escritura física del archivo en Drive
try:
    with open(ruta_readme, 'w', encoding='utf-8') as f:
        f.write(contenido_readme)
    print(f"✅ ¡El archivo README.md unificado se ha guardado con éxito en: {ruta_readme}!")
except Exception as e:
    print(f"❌ Error al escribir el archivo: {e}")


🔌 Conectando con Google Drive para actualizar documentación...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ ¡El archivo README.md unificado se ha guardado con éxito en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/README.md!
